# Transcribing Archival Images with Multimodal AI: A Gemini API Workflow

This notebook accompanies the **Digital History Pedagogy Toolkit** tutorial.

We use the Gemini API as a worked example, but the broader method can be adapted to other multimodal AI systems.

The notebook is written as **teaching material**. Before each code cell, you will find an explanation of what we are trying to accomplish, what the important pieces of code mean, and why the step matters.

> **Historical caution:** Treat AI output as a transcription candidate, not as ground truth.

**Last reviewed:** September 2026


## 1. What Are We Doing?

We begin with an archival image stored on your computer.

Python sends two things to Gemini:

1. the archival image;
2. a text prompt explaining how we want it transcribed.

Gemini sends a text response back to the notebook. Later, we repeat the process across several images and save the results.


## 2. Install the Libraries

We need `google-genai` to communicate with Gemini and `pandas` to create and save tables.

`%pip install` installs packages in the Python environment used by this notebook. `-U` means *upgrade if an older version is already installed*.


In [ ]:
%pip install -U google-genai pandas

Now import the tools we need.

- `genai` communicates with Gemini.
- `getpass` lets us enter the API key without displaying it.
- `Path` handles file paths.
- `datetime` records when an output was produced.
- `base64` prepares image bytes for the API request.
- `time` lets us pause between requests.
- `pandas` builds and saves tables.

You do not need to memorize these imports.


In [ ]:
from google import genai
from getpass import getpass
from pathlib import Path
from datetime import datetime, timezone
import base64
import time
import pandas as pd

## 3. Enter the Gemini API Key

An API key identifies the project making the request.

We use `getpass()` so the key is entered interactively rather than written visibly into the notebook.


In [ ]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

Now create a **client**. The client is the Python object that communicates with the Gemini API.


In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

## 4. Choose a Model

We save the model name in one variable so it is easy to change later.

At the time this notebook was reviewed, `gemini-3.7-flash` supports image input and is available on the Gemini API Free Tier. Availability may change.


In [ ]:
MODEL = "gemini-3.7-flash"

## 5. Point Python to the Archival Image

The notebook expects a folder called `archival_images` and a JPG file called `page_01.jpg`.

The path `archival_images/page_01.jpg` means: open the `archival_images` folder, then find `page_01.jpg`.


In [ ]:
IMAGE_PATH = Path("archival_images/page_01.jpg")

Check that Python can actually find the file. If the file is missing, the next cell gives a clear error.


In [ ]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Could not find {IMAGE_PATH}")

## 6. Create a Reusable Transcription Function

A **function** is a reusable block of code.

Our function receives an image path and a prompt. It reads the image, converts the image bytes into Base64, sends the prompt and image to Gemini, and returns Gemini's text response.

Base64 is simply a text-safe representation of binary image data. You do not need to memorize that conversion line.


In [ ]:
def transcribe_image(image_path, prompt, model=MODEL):
    image_path = Path(image_path)

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    interaction = client.interactions.create(
        model=model,
        input=[
            {"type": "text", "text": prompt},
            {
                "type": "image",
                "data": image_b64,
                "mime_type": "image/jpeg"
            }
        ]
    )

    return interaction.output_text

Once the function exists, we can transcribe an image with the simpler instruction:

`transcribe_image(IMAGE_PATH, SOME_PROMPT)`


## 7. First Attempt: A Minimal Prompt

We deliberately start with a short prompt. Saving it in a variable lets us reuse exactly the same wording later.


In [ ]:
MINIMAL_PROMPT = """
Transcribe all text visible in this archival document image.
Return only the transcription.
"""

Now send the image and prompt to Gemini. The returned text is stored in `minimal_transcription`.


In [ ]:
minimal_transcription = transcribe_image(
    IMAGE_PATH,
    MINIMAL_PROMPT
)

print(minimal_transcription)

Compare the result against the source image. Check for omissions, spelling changes, expanded abbreviations, missing marginalia, guessed text, and reading-order problems.


## 8. Second Attempt: An Archival Transcription Prompt

Now we make the transcription policy more explicit. The goal is not to assume a longer prompt is automatically better, but to test whether explicit archival rules change the model's behavior.


In [ ]:
ARCHIVAL_PROMPT = """
Transcribe the visible text in this archival document image as faithfully as possible.

Follow these rules:

1. Preserve visible spelling, capitalization, punctuation, and abbreviations where legible.
2. Do not modernize spelling, names, or place names.
3. Do not silently expand abbreviations.
4. Preserve line breaks and reading order where they are meaningful.
5. Include marginalia, stamps, seals, headings, and handwritten or typed annotations when visible.
6. If text is illegible, write [illegible].
7. If you can suggest a reading but are uncertain, write [uncertain: your reading].
8. Do not reconstruct missing or obscured text from context.
9. Do not add explanations, summaries, or historical commentary.
10. Return only the transcription.
"""

Run the same image again, changing only the prompt. The result is stored separately as `archival_transcription`.


In [ ]:
archival_transcription = transcribe_image(
    IMAGE_PATH,
    ARCHIVAL_PROMPT
)

print(archival_transcription)

## 9. Compare the Two Outputs

We now have two transcriptions of the **same image**. A pandas DataFrame lets us place them in one table for comparison.


In [ ]:
comparison_df = pd.DataFrame([
    {"prompt_type": "minimal", "transcription": minimal_transcription},
    {"prompt_type": "archival", "transcription": archival_transcription}
])

comparison_df

Do not judge the outputs by fluency alone. Compare both against the source image.


## 10. Evaluate the Transcriptions

Use a simple 2/1/0 rubric:

- **2 = strong**
- **1 = mixed**
- **0 = weak**

The next cell creates an empty evaluation table. The model does **not** fill in the scores; you do after reviewing the image.


In [ ]:
evaluation_categories = [
    "general_accuracy",
    "named_entities",
    "dates_references",
    "abbreviations",
    "layout_reading_order",
    "marginalia_stamps_annotations",
    "uncertainty_marking",
    "omissions",
    "unsupported_reconstruction"
]

evaluation_df = pd.DataFrame({
    "category": evaluation_categories,
    "minimal_score": "",
    "archival_score": "",
    "notes": ""
})

evaluation_df

The `notes` column matters as much as the score. The rubric organizes human judgment; it does not replace it.


## 11. Save the Outputs and Record How They Were Produced

Saving only the transcription is not enough for a reproducible workflow. We also want to record the image, prompt, model, and processing time.


In [ ]:
processed_at = datetime.now(timezone.utc).isoformat()

Create a table with one row for each transcription.


In [ ]:
results_df = pd.DataFrame([
    {
        "image_file": IMAGE_PATH.name,
        "prompt_type": "minimal",
        "prompt": MINIMAL_PROMPT.strip(),
        "model": MODEL,
        "processed_at_utc": processed_at,
        "transcription": minimal_transcription
    },
    {
        "image_file": IMAGE_PATH.name,
        "prompt_type": "archival",
        "prompt": ARCHIVAL_PROMPT.strip(),
        "model": MODEL,
        "processed_at_utc": processed_at,
        "transcription": archival_transcription
    }
])

results_df

Save the table as CSV. `index=False` prevents an extra numbered column, and `utf-8-sig` helps preserve multilingual characters in common spreadsheet software.


In [ ]:
results_df.to_csv(
    "transcription_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved transcription_comparison.csv")

## 12. Process Several Images

Once the workflow works for one page, we can apply the **same archival prompt** automatically to every JPG image in a folder.

For this teaching exercise, we use JPG only so the file-search code stays simple.


### 12.1 Find the Images

`Path("archival_images")` identifies the folder.

`.glob("*.jpg")` means: find every file in that folder whose name ends in `.jpg`.

The `*` is a wildcard meaning *any filename*. `sorted(...)` puts the files in alphabetical order.


In [ ]:
IMAGE_FOLDER = Path("archival_images")

image_files = sorted(
    IMAGE_FOLDER.glob("*.jpg")
)

print(f"Found {len(image_files)} image(s).")

Display the list so you can check which files Python found.


In [ ]:
image_files

### 12.2 Create an Empty Results List

The next line creates an empty Python list. As each image is processed, we will add one result record to this list.


In [ ]:
batch_results = []

### 12.3 Loop Through the Images

A `for` loop repeats the same code for every image in `image_files`.

Read the next block as a sequence:

1. take one image;
2. print its filename;
3. send it to Gemini;
4. save the result;
5. if something fails, record the error;
6. wait briefly;
7. move to the next image.


In [ ]:
for image_path in image_files:
    print(f"Transcribing: {image_path.name}")

    try:
        transcription = transcribe_image(
            image_path,
            ARCHIVAL_PROMPT
        )

        batch_results.append({
            "image_file": image_path.name,
            "prompt_type": "archival",
            "prompt": ARCHIVAL_PROMPT.strip(),
            "model": MODEL,
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            "transcription": transcription,
            "review_status": "",
            "review_notes": ""
        })

    except Exception as e:
        batch_results.append({
            "image_file": image_path.name,
            "prompt_type": "archival",
            "prompt": ARCHIVAL_PROMPT.strip(),
            "model": MODEL,
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            "transcription": "",
            "review_status": "API error",
            "review_notes": str(e)
        })

    time.sleep(2)

`try` means *try to process this image normally*. `except` means *if that fails, record the error instead of stopping the whole batch*.

`time.sleep(2)` pauses for two seconds before the next request. This is a simple teaching precaution, not a universal rate-limit rule.


### 12.4 Convert the Results into a Table

`batch_results` is a Python list. Turn it into a pandas DataFrame so each image becomes one row.


In [ ]:
batch_df = pd.DataFrame(batch_results)

batch_df

### 12.5 Save the Batch Results

Save the table as `archival_transcriptions.csv`. The empty review fields are there so a human reviewer can later record whether each transcription has been checked.


In [ ]:
batch_df.to_csv(
    "archival_transcriptions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved archival_transcriptions.csv")

## 13. What Changes When We Scale Up?

Automation can reproduce the same model behavior across an entire corpus.

If a model systematically drops marginalia, normalizes spelling, mishandles names, or reconstructs uncertain text, those tendencies may become properties of the resulting dataset.

Before scaling, test a representative sample, identify recurring failure patterns, define acceptable transcription quality, and decide how much human review is required.


## 14. Adapting the Workflow to Other Models

The Gemini-specific Python code can be replaced with another provider's SDK.

The historical workflow remains:

`authenticate → image → prompt → output → record process → evaluate`

The provider-specific code is replaceable. Decisions about fidelity, uncertainty, provenance, and evaluation are not.
